# NeoOLAF × RAGTree — Evaluate completed EventStoryLine + RESUME FinCausal

This notebook is intentionally specialized for the state left by the previous full parallel experiment.

## What it does

1. Opens the **existing** full-run state:
   `runs/full_process_isolated_eventstoryline_fincausal_v1`
2. Verifies that **EventStoryLine is already complete** and evaluates it **offline** from the persisted per-document results.
3. Evaluates whatever FinCausal results already exist as a **partial diagnostic**.
4. Launches **only pending FinCausal documents**.
5. Uses the same validated FinCausal parallelism:
   - **5 independent document processes**
   - **1 layer worker per process**
6. Reuses the same `full_progress.json`, so already-successful FinCausal records are never paid for again.
7. Appends a new UTC OpenRouter usage session for this resumed paid run.
8. Produces final EventStoryLine + FinCausal aggregate results.

### Important

**This notebook never launches EventStoryLine.**  
EventStoryLine is treated as frozen/completed and is evaluation-only here.

Use a fresh kernel and run from the top.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, subprocess, textwrap, re, hashlib, uuid as _uuid
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError("NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate

MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180

FINCAUSAL_DOCUMENT_WORKERS = 5
FINCAUSAL_LAYER_WORKERS = 1
MAX_FAILURES_PER_INVOCATION = 3
SUMMARY_CHECKPOINT_EVERY = 10

RUN_PAID_FINCAUSAL = True

RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "full_process_isolated_eventstoryline_fincausal_v1"
PROGRESS_PATH = RUNS_ROOT / "full_progress.json"
SUMMARY_PATH = RUNS_ROOT / "full_summary_live.json"
USAGE_SESSIONS_PATH = RUNS_ROOT / "openrouter_usage_sessions.json"
CONSOLE_LOG_ROOT = RUNS_ROOT / "_worker_console_logs" / "fincausal_resume"
CONSOLE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

if not RUNS_ROOT.is_dir():
    raise FileNotFoundError(
        "Existing full-run directory not found. This notebook is RESUME-ONLY:\n"
        f"{RUNS_ROOT}"
    )
if not PROGRESS_PATH.is_file():
    raise FileNotFoundError(
        "Existing full_progress.json not found. Refusing to start a new experiment."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Existing RUNS_ROOT:", RUNS_ROOT)
print("Python child executable:", sys.executable)
print("RUN_PAID_FINCAUSAL:", RUN_PAID_FINCAUSAL)
print("FinCausal parallelism: 5 document processes × 1 layer worker")


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Existing RUNS_ROOT: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1
Python child executable: c:\Users\galencarmedeiro\NeoOLAF\.venv\Scripts\python.exe
RUN_PAID_FINCAUSAL: True
FinCausal parallelism: 5 document processes × 1 layer worker


## Load normalized datasets and hard-check the existing experiment state

This cell makes **zero API calls**.

It refuses to continue if:
- the dataset sizes changed;
- the model/version differs from the original full run;
- the full dataset record-key sequence changed;
- EventStoryLine is not actually 443/443 complete.


In [2]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)

dataset_rows = {
    "eventstoryline": expstate.read_jsonl(DATASET_FILES["eventstoryline"]),
    "fincausal": expstate.read_jsonl(DATASET_FILES["fincausal"]),
}

assert len(dataset_rows["eventstoryline"]) == 443
assert len(dataset_rows["fincausal"]) == 967

progress = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))

assert progress["model"] == MODEL_NAME, (progress.get("model"), MODEL_NAME)
assert progress["host"] == OPENROUTER_HOST, progress.get("host")
assert progress["dataset_order"] == ["eventstoryline", "fincausal"]

EXPECTED_VERSIONS = {
    "eventstoryline": "v1.7",
    "fincausal": "unified-v1.3.1-selection-hotfix",
}

def record_keys(dataset_key):
    return [expstate.record_key(dataset_key, r) for r in dataset_rows[dataset_key]]

for k in ["eventstoryline", "fincausal"]:
    ds = progress["datasets"][k]
    assert ds["version"] == EXPECTED_VERSIONS[k]
    assert ds["total_records"] == len(dataset_rows[k])
    current_sha = hashlib.sha256(
        "\n".join(record_keys(k)).encode("utf-8")
    ).hexdigest()
    assert ds["record_keys_sha"] == current_sha, (
        k, "normalized full-dataset record-key sequence changed"
    )

esl_state = progress["datasets"]["eventstoryline"]
fc_state = progress["datasets"]["fincausal"]

# Strong protection: this resume notebook must NEVER pay for EventStoryLine again.
assert len(esl_state["completed_record_keys"]) == 443, (
    "EventStoryLine is not complete; this specialized notebook refuses to run.",
    len(esl_state["completed_record_keys"]),
)
assert esl_state.get("fully_finished_at_utc"), esl_state

assert fc_state["document_workers"] == FINCAUSAL_DOCUMENT_WORKERS
assert fc_state["layer_workers"] == FINCAUSAL_LAYER_WORKERS

print("EXISTING EXPERIMENT STATE")
print("=========================")
print("EventStoryLine:", len(esl_state["completed_record_keys"]), "/ 443 COMPLETE")
print(
    "FinCausal     :",
    len(fc_state["completed_record_keys"]),
    "/ 967 completed | pending =",
    967 - len(fc_state["completed_record_keys"]),
)
print("Previously recorded unresolved FinCausal failures:", len(fc_state.get("failures") or {}))
print("\nResume-state integrity: OK")
print("No API calls made.")


EXISTING EXPERIMENT STATE
EventStoryLine: 443 / 443 COMPLETE
FinCausal     : 5 / 967 completed | pending = 962
Previously recorded unresolved FinCausal failures: 5

Resume-state integrity: OK
No API calls made.


## Offline evaluator for completed results

This evaluates from persisted `posthoc_evaluation.json` files only.

It gives:
- EventStoryLine final full result;
- current FinCausal partial result before resume;
- final FinCausal full result after resume.

No model calls occur in this cell.


In [3]:
def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def atomic_json(path, obj):
    expstate.atomic_write_json(Path(path), obj)

def result_path(dataset_key, rkey):
    return (
        RUNS_ROOT
        / dataset_key
        / "full"
        / safe_dir_name(rkey)
        / "posthoc_evaluation.json"
    )

def _count(m, *names):
    if not isinstance(m, dict):
        return 0
    for name in names:
        if name in m and m[name] is not None:
            return int(m[name] or 0)
    return 0

def normalize_counts(m):
    tp = _count(m, "tp", "true_positive")
    fp = _count(m, "fp", "false_positive")
    fn = _count(m, "fn", "false_negative")
    pred = _count(m, "pred", "predicted")
    gold = _count(m, "gold", "gold_unique")
    if pred == 0 and tp + fp:
        pred = tp + fp
    if gold == 0 and tp + fn:
        gold = tp + fn
    return {"pred": pred, "gold": gold, "tp": tp, "fp": fp, "fn": fn}

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1

def load_completed_results(dataset_key):
    completed = set(progress["datasets"][dataset_key].get("completed_record_keys") or [])
    rows = []
    missing = []
    for source in dataset_rows[dataset_key]:
        rkey = expstate.record_key(dataset_key, source)
        if rkey not in completed:
            continue
        p = result_path(dataset_key, rkey)
        if not p.is_file():
            missing.append((rkey, str(p)))
            continue
        rows.append(json.loads(p.read_text(encoding="utf-8")))
    if missing:
        raise RuntimeError(
            f"{dataset_key}: progress says completed but result file is missing: {missing[:5]}"
        )
    return rows

def aggregate(dataset_key):
    rows = load_completed_results(dataset_key)
    rel_counts = [normalize_counts(r.get("relation_metrics") or {}) for r in rows]
    ep_counts = [normalize_counts(r.get("endpoint_metrics") or {}) for r in rows]

    rel_tp = sum(x["tp"] for x in rel_counts)
    rel_fp = sum(x["fp"] for x in rel_counts)
    rel_fn = sum(x["fn"] for x in rel_counts)
    rel_pred = sum(x["pred"] for x in rel_counts)
    rel_gold = sum(x["gold"] for x in rel_counts)
    rel_p, rel_r, rel_f1 = prf(rel_tp, rel_fp, rel_fn)

    ep_tp = sum(x["tp"] for x in ep_counts)
    ep_fp = sum(x["fp"] for x in ep_counts)
    ep_fn = sum(x["fn"] for x in ep_counts)
    ep_pred = sum(x["pred"] for x in ep_counts)
    ep_gold = sum(x["gold"] for x in ep_counts)
    ep_p, ep_r, ep_f1 = prf(ep_tp, ep_fp, ep_fn)

    doc_f1s = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r in rows
    ]
    positive_f1s = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r, c in zip(rows, rel_counts)
        if c["gold"] > 0
    ]

    ds = progress["datasets"][dataset_key]
    completed = len(ds.get("completed_record_keys") or [])
    total = ds["total_records"]

    return {
        "dataset": dataset_key,
        "version": ds["version"],
        "status": "COMPLETE" if completed == total else "PARTIAL",
        "completed_records": completed,
        "total_records": total,
        "pending_records": total - completed,
        "recorded_failures": len(ds.get("failures") or {}),
        "relation": {
            "pred": rel_pred,
            "gold": rel_gold,
            "tp": rel_tp,
            "fp": rel_fp,
            "fn": rel_fn,
            "precision": rel_p,
            "recall": rel_r,
            "micro_f1": rel_f1,
            "macro_doc_f1": sum(doc_f1s) / len(doc_f1s) if doc_f1s else 0.0,
            "macro_positive_gold_doc_f1": (
                sum(positive_f1s) / len(positive_f1s)
                if positive_f1s else 0.0
            ),
            "positive_gold_docs_completed": len(positive_f1s),
        },
        "endpoint": {
            "pred": ep_pred,
            "gold": ep_gold,
            "tp": ep_tp,
            "fp": ep_fp,
            "fn": ep_fn,
            "precision": ep_p,
            "recall": ep_r,
            "micro_f1": ep_f1,
        },
    }

def export_completed(dataset_key):
    path = RUNS_ROOT / f"{dataset_key}_completed_results.jsonl"
    rows = load_completed_results(dataset_key)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    tmp.replace(path)
    return path

def save_summary():
    summary = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "generated_at_utc": utc_now(),
        "datasets": {
            "eventstoryline": aggregate("eventstoryline"),
            "fincausal": aggregate("fincausal"),
        },
    }
    atomic_json(SUMMARY_PATH, summary)
    return summary

pre_resume_summary = save_summary()

print("EVENTSTORYLINE FINAL OFFLINE EVALUATION")
pprint(pre_resume_summary["datasets"]["eventstoryline"])

print("\nFINCAUSAL CURRENT PARTIAL EVALUATION")
pprint(pre_resume_summary["datasets"]["fincausal"])

print("\nNo API calls made by evaluator.")


EVENTSTORYLINE FINAL OFFLINE EVALUATION
{'completed_records': 443,
 'dataset': 'eventstoryline',
 'endpoint': {'fn': 2559,
              'fp': 7,
              'gold': 5192,
              'micro_f1': 0.6723697650663942,
              'precision': 0.9973484848484848,
              'pred': 2640,
              'recall': 0.5071263482280431,
              'tp': 2633},
 'pending_records': 0,
 'recorded_failures': 0,
 'relation': {'fn': 8738,
              'fp': 5091,
              'gold': 9640,
              'macro_doc_f1': 0.11130832321990927,
              'macro_positive_gold_doc_f1': 0.11130832321990927,
              'micro_f1': 0.11539691677860935,
              'positive_gold_docs_completed': 443,
              'precision': 0.1505089270815952,
              'pred': 5993,
              'recall': 0.09356846473029046,
              'tp': 902},
 'status': 'COMPLETE',
 'total_records': 443,
 'version': 'v1.7'}

FINCAUSAL CURRENT PARTIAL EVALUATION
{'completed_records': 5,
 'dataset': 'finc

## FinCausal-only worker

The child worker:
- loads exactly one normalized FinCausal record by stable `record_key`;
- strips gold before the pipeline;
- runs the frozen FinCausal configuration;
- exposes gold only after Layer 12;
- saves `posthoc_evaluation.json` to the same original full-run directory.

EventStoryLine is not imported as an execution target and cannot be launched by this notebook.


In [4]:
RESUME_WORKER_SCRIPT = RUNS_ROOT / "_resume_fincausal_worker.py"

worker_code = r"""
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import argparse, os, sys, json, time, shutil, re, traceback

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

parser = argparse.ArgumentParser()
parser.add_argument("--project-root", required=True)
parser.add_argument("--record-key", required=True)
parser.add_argument("--run-root", required=True)
parser.add_argument("--model", required=True)
parser.add_argument("--host", required=True)
parser.add_argument("--reasoning-effort", default="minimal")
parser.add_argument("--max-tokens", type=int, default=8192)
parser.add_argument("--request-timeout", type=int, default=180)
args = parser.parse_args()

PROJECT_ROOT = Path(args.project_root).resolve()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"
for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1_8 as adapters

dataset_key = "fincausal"
rkey = args.record_key
run_root = Path(args.run_root).resolve()
run_dir = run_root / "fincausal" / "full" / safe_dir_name(rkey)
failure_path = run_dir / "worker_failure.json"

try:
    RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
    PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
    ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)
    DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)
    RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)

    rows = expstate.read_jsonl(DATASET_FILES["fincausal"])
    matches = [r for r in rows if expstate.record_key("fincausal", r) == rkey]
    if len(matches) != 1:
        raise RuntimeError(f"{rkey}: expected one FinCausal row, got {len(matches)}")
    gold_record = matches[0]

    cfg = {
        "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
        "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
        "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
        "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
        "version": "unified-v1.3.1-selection-hotfix",
    }
    ontology_path = RAW_ONTOLOGY_FILES["fincausal"]

    # Only pending/failed records are launched. Clean their partial directory before retry.
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    pre_gold_contract = expstate.gold_contract_summary("fincausal", gold_record)

    clean_record = expstate.strip_gold(gold_record)
    forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean_record)
    if forbidden:
        raise RuntimeError(f"Gold leakage in pipeline input: {forbidden}")

    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])

    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    if gold_path.exists():
        gold_path.unlink()

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    started = utc_now()
    t0 = time.perf_counter()

    final_state = adapters.run_native_pipeline_record(
        dataset_key="fincausal",
        project_root=PROJECT_ROOT,
        input_jsonl=input_path,
        ontology_path=ontology_path,
        profile_path=cfg["profile"],
        guidance_path=cfg["guidance"],
        task_guidance_path=cfg["task"],
        relation_catalog_path=cfg["catalog"],
        relation_aliases_path=cfg["aliases"],
        run_dir=run_dir,
        model_name=args.model,
        api_key=api_key,
        host=args.host,
        workers=1,
        max_tokens=args.max_tokens,
        request_timeout=args.request_timeout,
        reasoning_effort=args.reasoning_effort,
        verbose=True,
        clean_run_dir=False,
    )

    # Gold only AFTER Layer 12 returned.
    expstate.write_jsonl(
        gold_path,
        [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
    )
    result = adapters.evaluate_state("fincausal", final_state, gold_record)
    result.update({
        "dataset": "fincausal",
        "version": cfg["version"],
        "record_key": rkey,
        "document_id": gold_record.get("document_id"),
        "title": gold_record.get("title"),
        "pre_run_gold_contract": pre_gold_contract,
        "run_dir": str(run_dir),
        "started_at_utc": started,
        "finished_at_utc": utc_now(),
        "elapsed_seconds": time.perf_counter() - t0,
        "model": args.model,
        "layer_workers": 1,
        "process_id": os.getpid(),
        "gold_visible_to_pipeline": False,
    })

    expected_gold = int(pre_gold_contract["gold_target_relation_count"])
    evaluated_gold = int((result.get("relation_metrics") or {}).get("gold", 0) or 0)
    if expected_gold > 0 and evaluated_gold == 0:
        raise RuntimeError(
            f"FinCausal evaluator integrity error: controller expected {expected_gold} "
            f"gold CAUSE relation(s), evaluator saw 0."
        )

    adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    print("WORKER_SUCCESS", rkey, json.dumps(result.get("relation_metrics") or {}))
    sys.exit(0)

except Exception as exc:
    try:
        run_dir.mkdir(parents=True, exist_ok=True)
        payload = {
            "dataset": "fincausal",
            "record_key": rkey,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
            "failed_at_utc": utc_now(),
            "process_id": os.getpid(),
        }
        failure_path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
    except Exception:
        pass
    traceback.print_exc()
    sys.exit(1)
"""

RESUME_WORKER_SCRIPT.write_text(textwrap.dedent(worker_code), encoding="utf-8")
compile(
    RESUME_WORKER_SCRIPT.read_text(encoding="utf-8"),
    str(RESUME_WORKER_SCRIPT),
    "exec",
)

print("FinCausal-only resume worker:", RESUME_WORKER_SCRIPT)
print("Worker syntax check: OK")
print("No API calls made.")


FinCausal-only resume worker: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\_resume_fincausal_worker.py
Worker syntax check: OK
No API calls made.


## Resume FinCausal

This is the only paid cell.

It starts from the current persistent state. For example, if the progress file says `5/967`, it launches only the remaining `962` records, including any previously failed unresolved records.

The five previously successful records remain untouched.


In [5]:
def write_usage_sessions():
    payload = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "host": OPENROUTER_HOST,
        "usage_sessions": progress.get("usage_sessions") or [],
        "note": (
            "Each resume execution has its own UTC window. "
            "Use these intervals with model=openai/gpt-oss-20b in OpenRouter."
        ),
    }
    atomic_json(USAGE_SESSIONS_PATH, payload)
    return payload

def start_resume_session():
    session = {
        "session_id": _uuid.uuid4().hex,
        "kind": "resume_fincausal_only",
        "started_at_utc": utc_now(),
        "finished_at_utc": None,
        "datasets": {
            "fincausal": {
                "started_at_utc": utc_now(),
                "finished_at_utc": None,
                "document_workers": 5,
                "layer_workers": 1,
                "completed_at_start": len(progress["datasets"]["fincausal"]["completed_record_keys"]),
                "completed_at_end": None,
            }
        },
    }
    progress.setdefault("usage_sessions", []).append(session)
    atomic_json(PROGRESS_PATH, progress)
    write_usage_sessions()
    return session

def finish_resume_session(session):
    fc = session["datasets"]["fincausal"]
    if fc.get("finished_at_utc") is None:
        fc["finished_at_utc"] = utc_now()
    fc["completed_at_end"] = len(progress["datasets"]["fincausal"]["completed_record_keys"])
    if session.get("finished_at_utc") is None:
        session["finished_at_utc"] = utc_now()
    atomic_json(PROGRESS_PATH, progress)
    write_usage_sessions()

def launch_fincausal_child(gold_record):
    rkey = expstate.record_key("fincausal", gold_record)

    log_path = CONSOLE_LOG_ROOT / f"{safe_dir_name(rkey)}.log"
    log_handle = open(log_path, "w", encoding="utf-8", buffering=1)

    cmd = [
        sys.executable,
        str(RESUME_WORKER_SCRIPT),
        "--project-root", str(PROJECT_ROOT),
        "--record-key", rkey,
        "--run-root", str(RUNS_ROOT),
        "--model", MODEL_NAME,
        "--host", OPENROUTER_HOST,
        "--reasoning-effort", REASONING_EFFORT,
        "--max-tokens", str(MAX_TOKENS),
        "--request-timeout", str(REQUEST_TIMEOUT),
    ]

    env = os.environ.copy()
    env["NEOOLAF_PROJECT_ROOT"] = str(PROJECT_ROOT)

    proc = subprocess.Popen(
        cmd,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env,
        cwd=str(PROJECT_ROOT),
    )

    return {
        "proc": proc,
        "record_key": rkey,
        "document_id": gold_record.get("document_id"),
        "title": gold_record.get("title"),
        "log_handle": log_handle,
        "log_path": log_path,
        "started_monotonic": time.monotonic(),
    }

def resume_fincausal():
    ds = progress["datasets"]["fincausal"]
    total = ds["total_records"]
    completed = set(ds.get("completed_record_keys") or [])

    pending_rows = [
        r for r in dataset_rows["fincausal"]
        if expstate.record_key("fincausal", r) not in completed
    ]

    if not pending_rows:
        print("FinCausal already complete 967/967. No API calls needed.")
        if ds.get("fully_finished_at_utc") is None:
            ds["fully_finished_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
        return True

    print(
        f"RESUME FINCAUSAL: completed={len(completed)}/{total} | "
        f"pending={len(pending_rows)} | processes=5 | layer_workers/process=1"
    )

    queue = iter(pending_rows)
    active = {}
    failures_this_invocation = 0
    stop_launching = False
    last_status_print = 0.0

    def launch_next():
        nonlocal stop_launching
        if stop_launching:
            return False
        try:
            row = next(queue)
        except StopIteration:
            return False

        child = launch_fincausal_child(row)
        active[child["record_key"]] = child
        print(
            f"START pid={child['proc'].pid} | {child['record_key']} | "
            f"completed={len(ds['completed_record_keys'])}/{total}"
        )
        return True

    for _ in range(FINCAUSAL_DOCUMENT_WORKERS):
        if not launch_next():
            break

    try:
        while active:
            now = time.monotonic()
            finished = []

            for rkey, info in list(active.items()):
                rc = info["proc"].poll()
                if rc is None:
                    continue

                info["log_handle"].close()
                finished.append(rkey)
                wall = time.monotonic() - info["started_monotonic"]

                if rc == 0 and result_path("fincausal", rkey).is_file():
                    if rkey not in ds["completed_record_keys"]:
                        ds["completed_record_keys"].append(rkey)
                    ds.setdefault("failures", {}).pop(rkey, None)
                    atomic_json(PROGRESS_PATH, progress)

                    print(
                        f"DONE pid={info['proc'].pid} | {rkey} | wall={wall:.1f}s | "
                        f"completed={len(ds['completed_record_keys'])}/{total}"
                    )

                    if (
                        len(ds["completed_record_keys"]) % SUMMARY_CHECKPOINT_EVERY == 0
                        or len(ds["completed_record_keys"]) == total
                    ):
                        current = save_summary()["datasets"]["fincausal"]
                        print(
                            f"CHECKPOINT FinCausal {current['completed_records']}/{total} | "
                            f"micro-F1={current['relation']['micro_f1']:.6f}"
                        )

                else:
                    failures_this_invocation += 1
                    failure_file = (
                        RUNS_ROOT
                        / "fincausal"
                        / "full"
                        / safe_dir_name(rkey)
                        / "worker_failure.json"
                    )
                    failure = {
                        "record_key": rkey,
                        "document_id": info.get("document_id"),
                        "title": info.get("title"),
                        "returncode": rc,
                        "log_path": str(info["log_path"]),
                        "failure_file": (
                            str(failure_file) if failure_file.is_file() else None
                        ),
                        "failed_at_utc": utc_now(),
                    }
                    if failure_file.is_file():
                        try:
                            failure["worker_failure"] = json.loads(
                                failure_file.read_text(encoding="utf-8")
                            )
                        except Exception:
                            pass

                    ds.setdefault("failures", {})[rkey] = failure
                    atomic_json(PROGRESS_PATH, progress)
                    save_summary()

                    print(
                        f"FAILED {rkey} | rc={rc} | "
                        f"failures_this_invocation={failures_this_invocation}/"
                        f"{MAX_FAILURES_PER_INVOCATION}\n"
                        f"  log: {info['log_path']}"
                    )

                    if failures_this_invocation >= MAX_FAILURES_PER_INVOCATION:
                        stop_launching = True
                        print(
                            "FAILURE BUDGET REACHED: no new documents will be launched. "
                            "Already-running children will finish."
                        )

            for rkey in finished:
                active.pop(rkey, None)

            while (
                not stop_launching
                and len(active) < FINCAUSAL_DOCUMENT_WORKERS
            ):
                if not launch_next():
                    break

            if active and now - last_status_print >= 30:
                print(
                    f"STATUS FinCausal: completed={len(ds['completed_record_keys'])}/{total} | "
                    f"active={len(active)} | failures={failures_this_invocation}"
                )
                last_status_print = now

            if active:
                time.sleep(1)

    except KeyboardInterrupt:
        print("\nKeyboardInterrupt: terminating current FinCausal child processes...")
        for info in active.values():
            try:
                info["proc"].terminate()
            except Exception:
                pass
        time.sleep(1)
        for info in active.values():
            try:
                if info["proc"].poll() is None:
                    info["proc"].kill()
            except Exception:
                pass
            try:
                info["log_handle"].close()
            except Exception:
                pass
        atomic_json(PROGRESS_PATH, progress)
        save_summary()
        raise

    completed_now = len(ds["completed_record_keys"])
    export_completed("fincausal")
    save_summary()

    if completed_now == total:
        ds["fully_finished_at_utc"] = utc_now()
        ds["failures"] = {}
        atomic_json(PROGRESS_PATH, progress)
        export_completed("fincausal")
        save_summary()
        print("\nFINCAUSAL FULL COMPLETE 967/967")
        return True

    print(
        f"\nFinCausal remains incomplete: {completed_now}/{total}. "
        "Rerun this notebook after resolving any new failure/credit issue."
    )
    return False

if not RUN_PAID_FINCAUSAL:
    print("RUN_PAID_FINCAUSAL=False -> resume stopped before API calls.")
else:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    # Reload progress immediately before paid work in case an earlier cell was rerun.
    progress = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))

    # Absolute guard against accidentally launching EventStoryLine.
    assert len(progress["datasets"]["eventstoryline"]["completed_record_keys"]) == 443

    session = start_resume_session()
    try:
        fin_complete = resume_fincausal()
    finally:
        finish_resume_session(session)

    if not fin_complete:
        raise RuntimeError(
            "FinCausal resume stopped before 967/967. "
            "Successful documents are persisted and will be skipped on the next resume."
        )


RESUME FINCAUSAL: completed=5/967 | pending=962 | processes=5 | layer_workers/process=1
START pid=24928 | fincausal:5:b4da21809cefa5ad | completed=5/967
START pid=10368 | fincausal:6:80d5d6a1738c7b8e | completed=5/967
START pid=11436 | fincausal:7:904acf3764c24276 | completed=5/967
START pid=5456 | fincausal:8:d670e6d50b3a53f9 | completed=5/967
START pid=3140 | fincausal:9:0b0a33f7388019b3 | completed=5/967
STATUS FinCausal: completed=5/967 | active=5 | failures=0
DONE pid=10368 | fincausal:6:80d5d6a1738c7b8e | wall=14.1s | completed=6/967
START pid=14556 | fincausal:10:7551d279863a4a5c | completed=6/967
DONE pid=24928 | fincausal:5:b4da21809cefa5ad | wall=19.2s | completed=7/967
START pid=15384 | fincausal:11:ffe2d4714fd469d5 | completed=7/967
DONE pid=14556 | fincausal:10:7551d279863a4a5c | wall=7.1s | completed=8/967
START pid=15828 | fincausal:12:4e8690e508dcacd0 | completed=8/967
DONE pid=11436 | fincausal:7:904acf3764c24276 | wall=22.3s | completed=9/967
DONE pid=3140 | fincausal

RuntimeError: FinCausal resume stopped before 967/967. Successful documents are persisted and will be skipped on the next resume.

## Final evaluation — zero API calls

Run this after FinCausal finishes (or at any time to inspect current progress).

It evaluates:
- **EventStoryLine 443/443** from the already completed full run;
- **FinCausal** from all successfully persisted results.


In [ ]:
# Reload latest persistent state first.
progress = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))

final_summary = save_summary()
export_completed("eventstoryline")
export_completed("fincausal")
usage = write_usage_sessions()

print("FINAL / CURRENT FULL BENCHMARK RESULTS")
print("======================================")

for k in ["eventstoryline", "fincausal"]:
    a = final_summary["datasets"][k]
    r = a["relation"]
    e = a["endpoint"]
    print(
        f"\n{k} [{a['version']}] "
        f"{a['completed_records']}/{a['total_records']} | {a['status']}"
    )
    print(
        f"relation: P={r['precision']:.6f} "
        f"R={r['recall']:.6f} "
        f"micro-F1={r['micro_f1']:.6f} "
        f"macro-positive-doc-F1={r['macro_positive_gold_doc_f1']:.6f} "
        f"TP={r['tp']} FP={r['fp']} FN={r['fn']}"
    )
    print(
        f"endpoint: P={e['precision']:.6f} "
        f"R={e['recall']:.6f} "
        f"micro-F1={e['micro_f1']:.6f}"
    )

print("\nOPENROUTER USAGE SESSIONS")
pprint(usage)

print("\nFiles:")
print(" -", SUMMARY_PATH)
print(" -", USAGE_SESSIONS_PATH)
print(" -", RUNS_ROOT / "eventstoryline_completed_results.jsonl")
print(" -", RUNS_ROOT / "fincausal_completed_results.jsonl")
print("\nYou can send me this executed notebook after completion.")
